# ПЗ-5. RAG: сравнение вопросов SberQuAD и RuBQ

В этой практической работе рассмотрим, как разные методы поиска влияют на ответы языковой модели и как качество поиска связано с качеством итогового ответа. На примере последовательно разберем:
- ответы модели без дополнительного контекста;
- поиск по общему корпусу SberQuAD и RuBQ с помощью BM25;
- семантический поиск с помощью E5-small;
- повторную сортировку найденных фрагментов с помощью реранкера;
- сравнение качества поиска и ответов на вопросах из двух наборов;
- сохранение параметров и результатов экспериментов в MLflow.

Главная цель практики: понять, как источник и формулировка вопроса влияют на поиск, почему найденный документ не всегда содержит нужный для ответа фрагмент и как оценивать вклад поиска в итоговый ответ модели.


Ноутбук использует локальную GGUF-модель через llama-cpp-python. Датасеты загружаются из Hugging Face, эксперименты сохраняются в локальном MLflow. В демонстрационном запуске используются 100 новостей. Ячейка установки ниже рассчитана только на Google Colab. Она ставит сборку llama-cpp-python с поддержкой CUDA 12.4 и подгружает библиотеки CUDA по путям, которые существуют только в окружении Colab (/usr/local/lib/python*/dist-packages). Поэтому при локальном запуске ее выполнять не нужно, а подготовку окружения следует сделать вручную. Для NVIDIA вместо сборки можно взять готовую сборку: pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124. Проверьте, что ускорение на GPU доступно:
```
from llama_cpp import llama_supports_gpu_offload
print(llama_supports_gpu_offload())
```
Если результат False, а видеокарта есть, библиотеки CUDA не найдены: проверьте установку CUDA и переменную LD_LIBRARY_PATH. Если GPU нет, в ячейках подключения моделей замените n_gpu_layers=-1 на n_gpu_layers=0 и уменьшите N_EVAL: на CPU генерация заметно медленнее.

In [ ]:
%pip install -q datasets huggingface_hub mlflow nltk rouge-score pandas matplotlib tqdm

%pip uninstall -y llama-cpp-python

%pip install -q --no-cache-dir \
  nvidia-cuda-runtime-cu12 \
  nvidia-cublas-cu12

%pip install -q --no-cache-dir \
  llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

import glob
import ctypes

names = [
    "libcudart.so.12",
    "libcublas.so.12",
    "libcublasLt.so.12",
]

for name in names:
    files = glob.glob(
        f"/usr/local/lib/python*/dist-packages/nvidia/**/{name}",
        recursive=True,
    )
    if files:
        ctypes.CDLL(files[0], mode=ctypes.RTLD_GLOBAL)

from llama_cpp import llama_supports_gpu_offload

print("GPU offload support:", llama_supports_gpu_offload())

## Зависимости и параметры

Установку выполняйте до импортов и после изменения версий перезапускайте kernel. `llama-cpp-python` устанавливается отдельно подходящей для вашей CUDA сборкой. Модели при первом запуске скачиваются; построение эмбеддингов полного корпуса может занять много времени и памяти.
`EMBEDDING_BATCH_SIZE` задает количество фрагментов в одном батче. При наличии свободной памяти GPU значение можно увеличить. При каждом запуске ячейки поиска эмбеддинги корпуса вычисляются заново.


In [1]:
# %pip install datasets rank-bm25 sentence-transformers pandas matplotlib mlflow tqdm

**Пояснение к результату.**

Ячейка содержит команду установки зависимостей в виде комментария. Если необходимые библиотеки уже установлены в текущем окружении, запускать установку не требуется. После изменения версий библиотек может потребоваться перезапуск kernel.


In [2]:
import hashlib, random, re, time
from collections import Counter
import numpy as np
import pandas as pd
import mlflow
from datasets import load_dataset, get_dataset_config_names
from tqdm.auto import tqdm

SEED = 42
EVAL_N_EASY = 100
EVAL_N_HARD = 100  # RuBQ содержит 169 вопросов; часть выделяется для validation
VALIDATION_SHARE = 0.3
CHUNK_SIZE, OVERLAP = 500, 100
TOP_K, CANDIDATE_K = 10, 50
RUBQ_TOP_K_VALUES = [5, 10, 20]  # сравнение на validation
MODEL_REPO = 'Qwen/Qwen3-8B-GGUF'
MODEL_FILENAME = 'Qwen3-8B-Q4_K_M.gguf'
MAX_TOKENS = 256
EMBEDDING_MODEL = 'intfloat/multilingual-e5-small'
QUERY_PREFIX, DOCUMENT_PREFIX = 'query: ', 'passage: '
EMBEDDING_BATCH_SIZE = 32  # увеличьте при наличии свободной памяти GPU
RERANKER_MODEL = 'BAAI/bge-reranker-v2-m3'
assert 0 < VALIDATION_SHARE < 1 and EVAL_N_EASY > 0 and EVAL_N_HARD > 0
assert 0 <= OVERLAP < CHUNK_SIZE and 1 <= TOP_K <= CANDIDATE_K
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('pz5_rag_mixed_corpora')

<Experiment: artifact_location='/media/project/susu_course/neural_gen/practices/mlruns/5', creation_time=1793360048928, effective_trace_archival_retention=None, experiment_id='5', last_update_time=1793360048928, lifecycle_stage='active', name='pz5_rag_mixed_corpora', tags={}, trace_location=None, workspace='default'>

**Пояснение к результату.**

Здесь задаются параметры эксперимента и настраивается MLflow. В сохраненном выводе создан или выбран эксперимент `pz5_rag_mixed_corpora`; это подтверждает настройку хранения запусков, но еще не является оценкой качества модели.


## Данные и постановка задачи

### Задача

В практике исследуется система вопросно-ответного поиска по документам (RAG, Retrieval-Augmented Generation). Система получает вопрос на русском языке, находит в коллекции текстов абзацы, которые могут содержать ответ, и передает их языковой модели, формирующей итоговый ответ. Поиск и генерация оцениваются отдельно: качество поиска показывает, были ли найдены нужные абзацы, качество ответа показывает, использовала ли модель найденное корректно. Так проще понять, где именно возникает ошибка.

Во всех экспериментах используются одношаговые вопросы: для ответа достаточно одного абзаца, и один поисковый запрос теоретически способен его найти. Вопросы, требующие последовательного поиска по нескольким документам, из выборки исключены.

### Используемые наборы данных

Практика опирается на два набора вопросов, которые различаются по сложности для поиска.

| Характеристика | SberQuAD | RuBQ 2.0 |
|---|---|---|
| Источник текстов | Русская Википедия | Русская Википедия |
| Происхождение вопросов | Вопросы составлены аннотаторами по конкретному абзацу | Вопросы к базе знаний (Wikidata), сформулированные независимо от текста абзаца |
| Вид ответа | Фрагмент абзаца | Название объекта базы знаний |
| Роль в практике | «Легкий» набор | «Сложный» набор |
| Отбор | Все вопросы с непустым контекстом и ответом | Только одношаговые вопросы с ровно одним эталонным объектом и абзацем, содержащим ответ |

**SberQuAD** - датасет для задачи извлечения ответа из текста. Поскольку вопросы формулировались при чтении абзаца, они, как правило, сильно пересекаются с ним по лексике. Для поисковой системы это благоприятный случай: даже простое сопоставление слов дает хороший результат.

**RuBQ 2.0** - датасет вопросов к базе знаний с привязанными к ним абзацами Википедии, в которых размечено наличие ответа. Вопрос и текст формулировались независимо, поэтому слова вопроса могут не совпадать со словами абзаца, а ответ может стоять в другой грамматической форме. Для поиска это более трудный случай, в котором становится заметно преимущество семантических моделей и реранкера. Для оценки ответа в качестве допустимых вариантов используются исходный ответ и название объекта в базе знаний, если они различаются.

### Почему используются два набора

Сравнение методов на одном наборе может привести к неверным выводам. Если вопросы лексически близки к тексту, любой метод поиска покажет высокое качество, и различия между ними окажутся малозаметными. Если же использовать только трудный набор, не будет опорной точки, по которой видно, насколько снижается качество при ухудшении условий. Два набора с одинаковой постановкой (один вопрос, один шаг поиска, ответ в Википедии) решают обе проблемы: поскольку тип задачи не меняется, разница в результатах объясняется различием самих данных, а не различием постановки.

### Общий корпус

Абзацы обоих наборов объединяются в единую коллекцию документов. Каждый документ получает идентификатор, равный хэшу SHA-256 его текста, поэтому одинаковые тексты не дублируются, а для каждого документа сохраняется список источников. Поиск по любому вопросу выполняется по всей коллекции, поэтому для вопросов одного набора абзацы другого набора играют роль отвлекающих документов. Это приближает условия к практическому сценарию, где коллекция неоднородна и содержит много материала, не относящегося к вопросу.

Для каждого вопроса хранится список идентификаторов абзацев, в которых содержится ответ. Именно по этому списку далее определяется, найден ли нужный документ.

In [3]:
documents, document_sources = {}, {}
def add_document(text, source):
    text = text.strip()
    doc_id = hashlib.sha256(text.encode('utf-8')).hexdigest()
    documents[doc_id] = text
    document_sources.setdefault(doc_id, set()).add(source)
    return doc_id

easy = []
sber = load_dataset('kuznetsoffandrey/sberquad', streaming=True)
for split, rows in sber.items():
    for row in tqdm(rows, desc=f'SberQuAD: {split}'):
        text = (row.get('context') or '').strip()
        if not text: continue
        doc_id = add_document(text, 'SberQuAD')
        af = row.get('answers') or {}
        answers = af.get('text', []) if isinstance(af, dict) else []
        if isinstance(answers, str): answers = [answers]
        answers = [a.strip() for a in answers if a and a.strip()]
        question = (row.get('question') or '').strip()
        if question and answers:
            easy.append({'question': question, 'answers': answers, 'document_ids': [doc_id],
                         'dataset': 'SberQuAD', 'source_split': split})

rubq_paragraphs = load_dataset('d0rj/RuBQ_2.0-paragraphs', streaming=True)
paragraph_ids = {}
for split, rows in rubq_paragraphs.items():
    for row in tqdm(rows, desc=f'RuBQ абзацы: {split}'):
        paragraph_id = str(row['uid'])
        text = (row.get('text') or '').strip()
        if not text: continue
        doc_id = add_document(text, 'RuBQ')
        if paragraph_id in paragraph_ids and paragraph_ids[paragraph_id] != doc_id:
            raise ValueError(f'Повторяющийся uid абзаца RuBQ с разными текстами: {paragraph_id}')
        paragraph_ids[paragraph_id] = doc_id

rubq = load_dataset('d0rj/RuBQ_2.0', streaming=True)
print('Исходные части RuBQ:', list(rubq))
hard, rubq_skipped = [], Counter()
for split, rows in rubq.items():
    for row in tqdm(rows, desc=f'RuBQ вопросы: {split}'):
        tags = row.get('tags') or []
        if '1-hop' not in tags or 'no_answer' in tags:
            rubq_skipped['не 1-hop или без ответа'] += 1; continue
        gold = row.get('answers') or []
        if len(gold) != 1:
            rubq_skipped['не один эталонный объект'] += 1; continue
        refs = (row.get('paragraphs_uids') or {}).get('with_answer') or []
        if not refs:
            rubq_skipped['нет абзаца с ответом'] += 1; continue
        missing = [str(x) for x in refs if str(x) not in paragraph_ids]
        if missing:
            rubq_skipped['отсутствует текст размеченного абзаца'] += 1; continue
        question = (row.get('question_text') or '').strip()
        answer = (row.get('answer_text') or '').strip()
        if not question or not answer:
            rubq_skipped['нет текста вопроса или ответа'] += 1; continue
        # answer_text — исходный ответ. label — другая запись того же единственного объекта.
        alternatives = [answer]
        label = (gold[0].get('label') or '').strip()
        if label and label.casefold() != answer.casefold(): alternatives.append(label)
        hard.append({'question': question, 'answers': alternatives,
                     'document_ids': sorted({paragraph_ids[str(x)] for x in refs}),
                     'dataset': 'RuBQ', 'source_split': split, 'question_id': str(row['uid'])})


print('Исключено вопросов RuBQ:', dict(rubq_skipped))
print('Документов в общем корпусе:', len(documents))
print('Документы по источникам:', {name: sum(name in src for src in document_sources.values()) for name in ('SberQuAD', 'RuBQ')})
print('Вопросов SberQuAD / RuBQ:', len(easy), '/', len(hard))
print('Пример RuBQ:', hard[0])


SberQuAD: train: 0it [00:00, ?it/s]

SberQuAD: validation: 0it [00:00, ?it/s]

SberQuAD: test: 0it [00:00, ?it/s]

README.md:   0%|          | 0.00/911 [00:00<?, ?B/s]

RuBQ абзацы: paragraphs: 0it [00:00, ?it/s]

README.md:   0%|          | 0.00/1.74k [00:00<?, ?B/s]

Исходные части RuBQ: ['test', 'dev']


RuBQ вопросы: test: 0it [00:00, ?it/s]

RuBQ вопросы: dev: 0it [00:00, ?it/s]

Исключено вопросов RuBQ: {'не один эталонный объект': 338, 'нет абзаца с ответом': 242, 'не 1-hop или без ответа': 1041}
Документов в общем корпусе: 70148
Документы по источникам: {'SberQuAD': 13489, 'RuBQ': 56773}
Вопросов SberQuAD / RuBQ: 50364 / 1289
Пример RuBQ: {'question': 'Кто написал роман «Хижина дяди Тома»?', 'answers': ['Г. Бичер-Стоу', 'Гарриет Бичер-Стоу'], 'document_ids': ['b30ffad8e8d2af52a7e8f74b1ae69e08b69851fbfa8427478cda70ab62047910'], 'dataset': 'RuBQ', 'source_split': 'test', 'question_id': '1'}


**Пояснение к результату.**

В сохраненном запуске загружено **70 148 уникальных текстов**: 13 489 из SberQuAD и 56 773 из RuBQ. Сумма по источникам больше общего числа, поскольку одинаковый текст может встречаться в обоих наборах и хранится один раз. Для оценки оставлено 50 364 вопроса SberQuAD и 1 289 вопросов RuBQ после заданной фильтрации.

Пример вопроса RuBQ связан с одним абзацем и имеет две формы эталонного ответа. Категории исключения подсчитываются последовательно: каждая исключенная запись попадает в первую подходящую причину.


## Разбиение вопросов на validation и test


Разбиение выполняется по-разному для двух наборов.

**SberQuAD.** В исходном датасете несколько вопросов могут относиться к одному абзацу. Если разделить вопросы случайно, вопросы к одному и тому же документу попадут и в validation, и в test, и параметры, подобранные на первой части, получат неявную подсказку для второй. Поэтому делится не список вопросов, а список документов: все вопросы к одному абзацу попадают в одну и ту же часть. Доля validation задается параметром `VALIDATION_SHARE`.

**RuBQ 2.0.** Здесь сохраняется исходное разбиение авторов набора: dev-часть служит для validation, test-часть для итоговой проверки. Это исключает произвольность при выборе вопросов.

Во всех случаях используется фиксированное значение `SEED`, поэтому разбиение воспроизводится при повторном запуске. Из каждой части затем берется случайная подвыборка (`EVAL_N_EASY` и `EVAL_N_HARD` вопросов), чтобы время расчета оставалось приемлемым.

Разбиение касается только вопросов. Коллекция документов остается общей: при поиске по любому вопросу доступны все абзацы обоих наборов, независимо от того, к какой части относятся вопросы.

Размер выборки ограничивает точность выводов. При нескольких десятках вопросов разница в один-два вопроса меняет метрику на несколько процентных пунктов, поэтому малые различия между методами не следует считать значимыми.

In [4]:
def split_by_group(rows, group_fn):
    groups = sorted({group_fn(x) for x in rows})
    random.Random(SEED).shuffle(groups)
    assert len(groups) >= 2
    nval = min(len(groups) - 1, max(1, round(len(groups) * VALIDATION_SHARE)))
    validation_groups = set(groups[:nval])
    val = [x for x in rows if group_fn(x) in validation_groups]
    test = [x for x in rows if group_fn(x) not in validation_groups]
    random.Random(SEED).shuffle(val); random.Random(SEED + 1).shuffle(test)
    return val, test

easy_val, easy_test = split_by_group(easy, lambda x: x['document_ids'][0])
rubq_splits = {item['source_split'] for item in hard}
dev_names = rubq_splits & {'dev', 'validation', 'train'}
if len(dev_names) != 1 or 'test' not in rubq_splits:
    raise ValueError(f'Не удалось определить исходные dev/test RuBQ: {rubq_splits}')
hard_val = [x for x in hard if x['source_split'] in dev_names]
hard_test = [x for x in hard if x['source_split'] == 'test']
random.Random(SEED).shuffle(hard_val); random.Random(SEED + 1).shuffle(hard_test)
assert hard_val and hard_test
validation_sets = {'SberQuAD': easy_val[:EVAL_N_EASY], 'RuBQ': hard_val[:EVAL_N_HARD]}
test_sets = {'SberQuAD': easy_test[:EVAL_N_EASY], 'RuBQ': hard_test[:EVAL_N_HARD]}
print('Validation:', {k: len(v) for k, v in validation_sets.items()})
print('Test:', {k: len(v) for k, v in test_sets.items()})


Validation: {'SberQuAD': 100, 'RuBQ': 100}
Test: {'SberQuAD': 100, 'RuBQ': 100}


**Пояснение к результату.**

Для validation и test выбрано по **100 вопросов из каждого набора**. SberQuAD разделен по контекстам; для RuBQ используются исходные части `dev` и `test`. Тексты обеих частей присутствуют в поисковом корпусе, чтобы система могла находить сведения для ответа; вопросы test не применяются при выборе настроек.


In [5]:
from llama_cpp import Llama

LLM = Llama.from_pretrained(
    repo_id=MODEL_REPO,
    filename=MODEL_FILENAME,
    n_ctx=12000,
    n_gpu_layers=-1,
    verbose=False,
)


def ask_llm(prompt: str, temperature: float = 0.0, max_tokens: int = MAX_TOKENS, llm=LLM) -> str:
    """Отправляет prompt локальной модели.

    Args:
        prompt: Инструкция и исходные данные.
        temperature: Степень случайности генерации.
        max_tokens: Максимальная длина ответа в токенах.
        llm: Экземпляр локальной модели.

    Returns:
        Текст ответа модели.
    """
    response = llm.create_chat_completion(
        messages=[{"role": "user", "content": f"{prompt}\n/no_think"}],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    answer = response["choices"][0]["message"]["content"] or ""
    answer = re.sub(r"<think>.*?</think>", "", answer, flags=re.DOTALL)
    answer = re.sub(r"<think>.*", "", answer, flags=re.DOTALL)
    return answer.strip()





/media/project/susu_course/neural_gen/.venv_pz4/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


**Пояснение к результату.**

Модель Qwen загружена для генерации ответов. Предупреждение в сохраненном выводе относится к устаревшему аргументу загрузки и само по себе не сообщает об ошибке инференса. Устройство и фактическую загрузку слоев на GPU по этому выводу определить нельзя.


## Метрики и процедура оценки

Качество оценивается двумя способами: насколько хорошо найдены нужные документы и насколько правилен итоговый ответ. Так проще понять, где именно возникает ошибка.

### Оценка ответа

Ответ модели сравнивается с эталонными вариантами. У вопроса может быть несколько допустимых ответов, поэтому берется лучший результат по всем вариантам. Перед сравнением текст приводится к нижнему регистру и разбивается на слова, знаки препинания отбрасываются.

- **EM (Exact Match)** равна 1, если последовательность слов ответа в точности совпадает с эталоном, и 0 в остальных случаях. Это строгая метрика: ответ «Он родился в Риме» при эталоне «Рим» получит 0.
- **F1** учитывает частичное совпадение. Считаются доля слов ответа, присутствующих в эталоне (precision), и доля слов эталона, присутствующих в ответе (recall), а затем их гармоническое среднее. Метрика менее строгая, чем EM, но по-прежнему сравнивает слова в точной форме: «Рим» и «Риме» считаются

In [6]:


def score_answer(prediction, answers):
    """Сравнивает ответ модели с эталонными вариантами по EM и F1.

    Args:
        prediction: Ответ языковой модели.
        answers: Непустой список допустимых эталонных ответов.

    Returns:
        Кортеж из лучших значений EM и F1 среди эталонов. Значения
        каждой метрики выбираются независимо.
    """
    pred = re.findall(r'\w+', prediction.lower())
    results = []
    for answer in answers:
        ref = re.findall(r'\w+', answer.lower())
        common = sum((Counter(pred) & Counter(ref)).values())
        precision = common / len(pred) if pred else 0
        recall = common / len(ref) if ref else 0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
        results.append((float(pred == ref), f1))
    return max(x[0] for x in results), max(x[1] for x in results)

def prompt_for(question, hits=None):
    """Составляет запрос к модели с найденными фрагментами или без них.

    Args:
        question: Вопрос пользователя.
        hits: Список найденных фрагментов с полем ``text``. Значение
            None задаёт режим ответа без контекста.

    Returns:
        Текст prompt с инструкцией, вопросом и, при наличии, контекстом.
    """
    if hits is None:
        return f'Ответь на вопрос максимально кратко. Если не знаешь, скажи «Не знаю».\nВопрос: {question}\nОтвет:'
    context = '\n\n'.join(f'Фрагмент {i}:\n{h["text"]}' for i,h in enumerate(hits, 1))
    return f'Ответь кратко, используя только контекст. Если ответа нет, скажи «Не знаю».\nКонтекст:\n{context}\nВопрос: {question}\nОтвет:'

def evaluate(items, search=None, k=TOP_K):
    """Оценивает ответы модели и результаты поиска на наборе вопросов.

    Args:
        items: Записи с вопросом, эталонными ответами, идентификаторами
            релевантных документов и названием датасета.
        search: Поисковый объект с методом ``search(question, k)`` или
            None для ответа без дополнительного контекста.
        k: Максимальное число фрагментов, возвращаемых поиском.

    Returns:
        DataFrame с ответом, EM, F1, временем и найденными фрагментами
        для каждого вопроса. Для поиска ``hit`` показывает наличие
        хотя бы одного релевантного документа, ``document_recall``
        долю найденных релевантных документов; без поиска они равны None.
    """
    rows = []
    for item in tqdm(items):
        start = time.perf_counter()
        hits = None if search is None else search.search(item['question'], k)
        answer = 'Не знаю' if hits == [] else ask_llm(prompt_for(item['question'], hits))
        em, f1 = score_answer(answer, item['answers'])
        found = set() if hits is None else {h['document_id'] for h in hits}
        relevant = set(item['document_ids'])
        rows.append({'dataset': item['dataset'], 'question': item['question'], 'answers': item['answers'],
                     'document_ids': item['document_ids'], 'answer': answer, 'em': em, 'f1': f1,
                     'hit': None if hits is None else int(bool(found & relevant)),
                     'document_recall': None if hits is None else len(found & relevant) / len(relevant),
                     'n_hits': 0 if hits is None else len(hits),
                     'empty_context': None if hits is None else int(not hits),
                     'context_chars': 0 if hits is None else sum(len(h['text']) for h in hits),
                     'seconds': time.perf_counter() - start, 'hits': hits})
    return pd.DataFrame(rows)

def metrics(df):
    """Усредняет показатели оценки по вопросам.

    Args:
        df: DataFrame с результатами функции ``evaluate``.

    Returns:
        Словарь со средними EM, F1, временем и размером контекста.
        Для результатов поиска также включает средние ``hit``,
        ``document_recall``, число фрагментов и долю пустых контекстов.
    """
    out = {k: float(df[k].mean()) for k in ('em', 'f1', 'seconds', 'context_chars')}
    if df['hit'].notna().any():
        out.update({k: float(df[k].mean()) for k in ('hit', 'document_recall', 'n_hits', 'empty_context')})
    return out

**Пояснение к результату.**

Функция `score_answer` сравнивает ответ с эталонами по словам и выбирает лучший EM и F1 среди вариантов. `evaluate` дополнительно сохраняет найденные фрагменты и время. `hit` означает попадание хотя бы одного размеченного документа, а `document_recall` , долю найденных размеченных документов; попадание документа не гарантирует, что найденный фрагмент содержит нужный факт.


### Один вопрос без контекста

Сравним два prompt на одном и том же вопросе из validation RuBQ. Сначала попросим модель ответить кратко, затем добавим правило «Не знаю». В обоих случаях модель получает только вопрос; эталон выводится для проверки, но не передается модели. Одного примера недостаточно, чтобы оценить качество prompt на всей выборке.


In [25]:
example_no_context = validation_sets['RuBQ'][0]
question_no_context = example_no_context['question']

prompt_simple = f'Ответь на вопрос максимально кратко.\nВопрос: {question_no_context}\nОтвет:'
answer_simple = ask_llm(prompt_simple)

print('Вопрос:', question_no_context)
print('Эталон:', example_no_context['answers'])
print('Ответ без контекста:', answer_simple)


Вопрос: Когда родился Омар Хайям?
Эталон: ['18 мая 1048 г', '1048-05-24T00:00:00Z']
Ответ без контекста: Омар Хайям родился в 1048 году.


В данном случае модель без внешнего контекста дала правильный ответ. Скорее всего часть необходимой информации была усвоена ею на этапе обучения и доступна при генерации ответа.

In [27]:
example_no_context = validation_sets['RuBQ'][85]
question_no_context = example_no_context['question']

prompt_simple = f'Ответь на вопрос максимально кратко.\nВопрос: {question_no_context}\nОтвет:'
answer_simple = ask_llm(prompt_simple)

print('Вопрос:', question_no_context)
print('Эталон:', example_no_context['answers'])
print('Ответ без контекста:', answer_simple)


Вопрос: Где захоронен Борис Годунов?
Эталон: ['Свято-Троицкая Сергиева Лавра, Сергиев Посад', 'Троице-Сергиева лавра']
Ответ без контекста: В Новгороде-Северском.


На вопрос о месте захоронения Бориса Годунова модель без правила отказа ответила «В Новгороде-Северском», что не совпадает с приведенными эталонами. На этом примере модель дала уверенный неправильный ответ, хотя контекст ей не передавался.


In [28]:
prompt_with_abstention = (f'Ответь на вопрос максимально кратко. '
                          f'Если не знаешь, скажи «Не знаю».\n'
                          f'Вопрос: {question_no_context}\nОтвет:')
answer_with_abstention = ask_llm(prompt_with_abstention)

print('Вопрос:', question_no_context)
print('Эталон:', example_no_context['answers'])
print('Ответ с правилом отказа:', answer_with_abstention)


Вопрос: Где захоронен Борис Годунов?
Эталон: ['Свято-Троицкая Сергиева Лавра, Сергиев Посад', 'Троице-Сергиева лавра']
Ответ с правилом отказа: Не знаю.



На тот же вопрос с дополнительным правилом модель ответила «Не знаю». Это более осторожный ответ на данном примере: модель не повторила прежнюю ошибку. Один пример не позволяет заключить, что инструкция улучшит качество на всей выборке; также важно учитывать возможные отказы на вопросы, на которые модель способна ответить.


## Базовый уровень: ответ без поиска

Прежде чем оценивать систему с поиском, нужно выяснить, что модель умеет без него. В этом режиме (closed-book) вопрос передается модели без контекста, и она отвечает только по знаниям, полученным при обучении. Результат служит точкой отсчета: качество системы с поиском имеет смысл оценивать только относительно него. Если модель и так отвечает правильно, поиск не добавляет ничего, а если ответ без контекста заметно хуже, разница показывает вклад найденных документов.




In [29]:
baseline_results = {}
baseline_rows = []
for group, sample in validation_sets.items():
    result = evaluate(sample, search=None)
    baseline_results[group] = result
    values = metrics(result)
    baseline_rows.append({'dataset': group, 'n_questions': len(sample), **values})
    with mlflow.start_run(run_name=f'validation_{group}_no_context'):
        mlflow.log_params({'dataset': group, 'split': 'validation', 'method': 'no_context',
                           'eval_n': len(sample), 'seed': SEED, 'model_repo': MODEL_REPO,
                           'model_filename': MODEL_FILENAME, 'max_tokens': MAX_TOKENS})
        mlflow.log_metrics(values)
        mlflow.log_text(prompt_for('{question}'), 'prompt.txt')
        mlflow.log_dict(result.to_dict(orient='records'), 'results.json')
        mlflow.log_dict({'questions': sample}, 'questions.json')

baseline_table = pd.DataFrame(baseline_rows)
print('Результаты без контекста:')
print(baseline_table[['dataset', 'n_questions', 'em', 'f1', 'seconds']].to_string(index=False))
for group, result in baseline_results.items():
    print(f'\n{group}: примеры ответов')
    print(result[['question', 'answers', 'answer', 'em', 'f1']].head(3).to_string(index=False))


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

Результаты без контекста:
 dataset  n_questions   em       f1  seconds
SberQuAD          100 0.01 0.070694 0.382178
    RuBQ          100 0.20 0.265048 0.216384

SberQuAD: примеры ответов
                                                                                question                                              answers                                                                 answer  em  f1
                                                            Что собой представляют гены? [Являются элементарными единицами наследственности.] Гены — это участки ДНК, несущие информацию о наследственных признаках. 0.0 0.0
                    Кто из французов забил в дополнительное время победный золотой гол ?                                      [Давид Трезеге]                                                               Не знаю. 0.0 0.0
От какой фирмы государственного монополиста зависела группа Машина времени до 1991 года?                                            [Мелодия]        

## Разбиение документов на фрагменты

Документы в коллекции могут быть длинными, а поисковая модель и языковая модель работают с ограниченным объемом текста. Кроме того, чем короче фрагмент, тем точнее его эмбеддинг отражает одну тему, а в контекст для генерации попадает меньше лишнего. Поэтому перед индексацией каждый документ делится на фрагменты (чанки) фиксированной длины в символах (`CHUNK_SIZE`). Соседние фрагменты перекрываются на `OVERLAP` символов: если важный факт оказался на границе, он целиком попадет хотя бы в один из двух фрагментов.

Для каждого фрагмента сохраняются три поля: идентификатор документа, идентификатор самого фрагмента и текст. Идентификатор документа нужен при оценке: поиск работает с фрагментами, а нужные документы заданы в разметке вопросов. Поэтому фрагмент считается попаданием, если он принадлежит одному из документов, содержащих ответ.

У такого способа разбиения есть ограничения, о которых нужно помнить при анализе результатов.

1. **Граница фрагмента может разделить ответ.** Если ответ длиннее перекрытия или стоит на стыке, ни один фрагмент не будет содержать его целиком. Тогда документ найден (`hit` = 1), но в переданном модели тексте ответа нет.
2. **Разбиение не учитывает структуру текста.** Граница может пройти посреди слова или предложения. Более аккуратные способы делят текст по предложениям или абзацам, но усложняют код.
3. **Размер фрагмента влияет на результат.** Короткие фрагменты повышают точность поиска, но чаще разрывают контекст, длинные сохраняют контекст, но добавляют шум и увеличивают объем запроса к модели. Если исходные абзацы короче `CHUNK_SIZE`, разбиения не происходит, и параметр ни на что не влияет.


In [ ]:
def make_chunks(docs, size=CHUNK_SIZE, overlap=OVERLAP):
    result = []
    for doc_id, text in docs.items():
        for start in range(0, len(text), size - overlap):
            result.append({'document_id': doc_id, 'chunk_id': f'{doc_id}:{start}', 'text': text[start:start + size]})
            if start + size >= len(text): break
    return result
chunks = make_chunks(documents)
print('Фрагментов:', len(chunks))

## Лексический поиск: BM25

Первый метод поиска служит базовым уровнем. BM25 оценивает, насколько фрагмент подходит к вопросу, по словам, которые они имеют общие. Оценка учитывает три фактора.


Оценка фрагмента $D$ для вопроса $Q$, состоящего из слов $q_1, \dots, q_n$:

$$
\mathrm{score}(D, Q) = \sum_{i=1}^{n} \mathrm{IDF}(q_i) \cdot \frac{f(q_i, D)\,(k_1 + 1)}{f(q_i, D) + k_1 \left(1 - b + b \cdot \dfrac{|D|}{\mathrm{avgdl}}\right)}
$$

где

- $f(q_i, D)$ , сколько раз слово $q_i$ встречается во фрагменте $D$;
- $|D|$ , длина фрагмента в словах, $\mathrm{avgdl}$ , средняя длина фрагмента в коллекции;
- $k_1$ , параметр насыщения частоты слова. Чем он меньше, тем быстрее дополнительные упоминания перестают увеличивать оценку;
- $b \in [0, 1]$ , степень учета длины. При $b = 0$ длина не учитывается, при $b = 1$ учитывается полностью;
- $\mathrm{IDF}(q_i)$ , вес редкости слова:

$$
\mathrm{IDF}(q_i) = \ln\frac{N - n(q_i) + 0{,}5}{n(q_i) + 0{,}5}
$$

где $N$ , число фрагментов в коллекции, $n(q_i)$ , число фрагментов, содержащих слово $q_i$.

Как читать формулу: для каждого слова вопроса берется его вес редкости (IDF) и умножается на дробь, которая растет с ростом числа упоминаний во фрагменте, но не безгранично (стремится к $k_1 + 1$) и уменьшается для длинных фрагментов. Затем вклады всех слов вопроса суммируются. Слова вопроса, которых нет во фрагменте, вклада не дают.
Перед индексацией текст приводится к нижнему регистру и разбивается на слова. Индекс строится один раз по всем фрагментам. При поиске вопрос разбивается так же, для каждого фрагмента вычисляется оценка, и возвращаются `k` фрагментов с наибольшими оценками вместе с их оценками.

Метод не требует обучения, работает быстро и хорошо находит документы, лексически близкие к вопросу. Его ограничения вытекают из принципа работы: сопоставляются только точно совпадающие слова. Метод не учитывает словоизменение («Рим» и «Риме» для него разные слова), синонимы и перефразирование. Поэтому следует ожидать, что BM25 лучше справится с вопросами, формулировка которых близка к тексту абзаца, и хуже с вопросами, где слова вопроса и текста различаются. Именно это различие исследуется на двух наборах данных, а результат BM25 служит точкой отсчета для семантического поиска и реранкера.

In [32]:
from rank_bm25 import BM25Okapi
def tokenize(text): return re.findall(r'\w+', text.lower())
class BM25Search:
    def __init__(self, chunks):
        self.chunks = chunks
        self.index = BM25Okapi([tokenize(x['text']) for x in chunks])
    def search(self, question, k=TOP_K):
        scores = self.index.get_scores(tokenize(question))
        return [{**self.chunks[i], 'score': float(scores[i])} for i in np.argsort(-scores, kind='stable')[:k]]
bm25_search = BM25Search(chunks)

### Пример ответа с поиском

После оценки без контекста подготовим фрагменты и индекс BM25 в ячейках выше. Теперь зададим по одному вопросу из каждого набора: найдем фрагменты в общем корпусе и передадим их модели. Для сравнения выведем ранее полученный ответ без контекста.

`Исходный документ найден` показывает попадание хотя бы одного размеченного документа, но нужный факт может находиться в другой его части. Просмотрите найденные фрагменты и проверьте, подтверждают ли они ответ.


In [34]:
idx = 2
for group in ('SberQuAD', 'RuBQ'):
    item = validation_sets[group][idx]
    baseline_answer = baseline_results[group].iloc[idx]['answer']
    hits = bm25_search.search(item['question'], k=TOP_K)
    rag_answer = 'Не знаю' if not hits else ask_llm(prompt_for(item['question'], hits))
    print(f'\n--- {group} ---')
    print('Вопрос:', item['question'])
    print('Эталон:', item['answers'])
    print('Без контекста:', baseline_answer)
    print('С поиском BM25:', rag_answer)
    print('Исходный документ найден:', any(h['document_id'] in item['document_ids'] for h in hits))
    for i, hit in enumerate(hits, 1):
        print(f"\nФрагмент {i} (оценка BM25: {hit['score']:.3f}):\n{hit['text']}")



--- SberQuAD ---
Вопрос: От какой фирмы государственного монополиста зависела группа Машина времени до 1991 года?
Эталон: ['Мелодия']
Без контекста: Не знаю.
С поиском BM25: Фирмы Мелодия.
Исходный документ найден: True

Фрагмент 1 (оценка BM25: 43.272):
на. В этом же году Александр Кутиков становится продюсером группы. Благодаря этому Машина времени больше не зависит от государственного монополиста фирмы Мелодия .

Фрагмент 2 (оценка BM25: 20.211):
23 апреля 1991 года группа принимает участие в Международном фестивале Музыканты мира детям Чернобыля в Минске, а также в Благотворительной акции солидарности с программой Взгляд . Вопреки своим убеждениям не принимать участия в политических акциях, Макаревич помогает творить историю России, выступая на баррикадах 20 августа 1991 года перед защитниками Белого дома, тем самым поддержав Бориса Ельцина. В этом же году Александр Кутиков становится продюсером группы. Благодаря этому Машина времени бо

Фрагмент 3 (оценка BM25: 19.836):
С 2009 г.

Влияние поиска на ответ было разным.

SberQuAD. Без контекста модель ответила «Не знаю». После поиска она назвала фирму «Мелодия», как в эталоне. Первый фрагмент содержит нужный факт: группа перестала зависеть от государственного монополиста после того, как Кутиков стал ее продюсером. Для этого примера поиск дал модели информацию, которой она не сообщила без контекста.
Также найденные фрагменты неодинаково полезны. Второй фрагмент обрывается как раз перед важной частью, третий и шестой посвящены той же группе, но не отвечают на вопрос, а десятый содержит лишь словосочетание «Виртуальная машина времени». BM25 нашел их по совпадающим словам, хотя они не помогают с ответом.

RuBQ. Модель ответила «Аристотель» и без поиска, и с ним. Первый найденный фрагмент представляет собой библиографическую запись «Аристотель. Поэтика», а третий и шестой содержат похожие записи. Они согласуются с ответом, но по этому примеру нельзя установить, использовала ли модель найденный текст или воспроизвела известный ей факт. Остальные результаты, например фрагмент о микробиальной фауне, ответа не содержат.

### Задание 1. Сравнение с ответами без поиска

На тех же validation-вопросах из каждого датасета самостоятельно оцените вариант с BM25 и сравните его с сохраненным базовым уровнем `baseline_results`.

1. Составьте таблицу с EM, F1 и средним временем для ответов без контекста и с поиском. Для BM25 дополнительно покажите `hit` и `document_recall`.
2. Разберите несколько примеров, где ответ изменился после добавления контекста. Есть ли во фрагментах информация, подтверждающая ответ?
3. Сделайте вывод: на каких вопросах поиск помог, а на каких нет? Учитывайте ограничения EM и F1 при сравнении разных форм ответа.


## Семантический поиск по эмбеддингам

В предыдущем разделе BM25 находил фрагменты по совпадающим словам. В примерах к полезным фрагментам добавлялись тексты, которые содержали те же слова, но не отвечали на вопрос. Кроме того, одна и та же мысль может быть выражена разными словами. Теперь рассмотрим другой подход: сравнение вопроса и фрагментов по их эмбеддингам.

### Как работает поиск

Тексты с близким содержанием могут получить близкие векторы, даже если их слова не совпадают полностью.
В практике используется модель `intfloat/multilingual-e5-small`, поддерживающая русский язык. При построении индекса к каждому фрагменту добавляется префикс `passage:`, а при обработке вопроса , `query:`. Эти префиксы показывают модели роль текста.
Для каждого фрагмента корпуса эмбеддинг вычисляется один раз. Полученные векторы образуют матрицу: каждая строка соответствует одному фрагменту из `chunks`. При поиске модель отдельно вычисляет вектор вопроса и сравнивает его со строками матрицы.

### Как измеряется близость

Для вектора вопроса $q$ и вектора фрагмента $d$ косинусное сходство определяется формулой:

$$
\operatorname{sim}(q, d) = \frac{q \cdot d}{\lVert q \rVert \lVert d \rVert}.
$$

Сходство измеряет близость направлений векторов. Чем больше значение, тем ближе векторные представления по выбранной модели. Перед сравнением векторы нормализуются до единичной длины (`normalize_embeddings=True`), поэтому косинусное сходство можно вычислить как скалярное произведение `matrix @ query`. Возвращаются `k` фрагментов с наибольшими оценками. Численные оценки E5 и BM25 имеют разные шкалы и напрямую не сравниваются.



In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder

class EmbeddingSearch:
    def __init__(self, chunks, model):
        self.chunks = chunks
        self.model = model
        # Каждой строке матрицы соответствует фрагмент в том же порядке.
        self.matrix = model.encode(
            [DOCUMENT_PREFIX + chunk['text'] for chunk in chunks],
            batch_size=EMBEDDING_BATCH_SIZE,
            normalize_embeddings=True,
            show_progress_bar=True,
        )

    def search(self, question, k=TOP_K):
        query = self.model.encode([QUERY_PREFIX + question], normalize_embeddings=True)[0]
        scores = self.matrix @ query
        indices = np.argsort(-scores, kind='stable')[:k]
        return [{**self.chunks[i], 'score': float(scores[i])} for i in indices]

embedding_model = SentenceTransformer(EMBEDDING_MODEL)
print('Устройство модели:', embedding_model.device)
semantic_search = EmbeddingSearch(chunks, embedding_model)


**Пояснение к результату.**

После загрузки модели выводится устройство, на котором она работает. Затем рассчитываются эмбеддинги всех фрагментов и создается матрица для семантического поиска. При повторном запуске ячейки эти вычисления выполняются заново.


### Пример ответа с семантическим поиском

Зададим те же вопросы, что и в примере с BM25: по одному из SberQuAD и RuBQ. E5-small найдет фрагменты в общем корпусе, после чего языковая модель ответит по ним. Сравните найденные тексты с выдачей BM25: присутствует ли в них информация для ответа и какие фрагменты оказались посторонними?


In [37]:
idx = 2
for group in ('SberQuAD', 'RuBQ'):
    item = validation_sets[group][idx]
    baseline_answer = baseline_results[group].iloc[idx]['answer']
    hits = semantic_search.search(item['question'], k=TOP_K)
    rag_answer = 'Не знаю' if not hits else ask_llm(prompt_for(item['question'], hits))
    print(f'\n--- {group} ---')
    print('Вопрос:', item['question'])
    print('Эталон:', item['answers'])
    print('Без контекста:', baseline_answer)
    print('С семантическим поиском:', rag_answer)
    print('Исходный документ найден:', any(h['document_id'] in item['document_ids'] for h in hits))
    for i, hit in enumerate(hits, 1):
        print(f"\nФрагмент {i} (сходство: {hit['score']:.3f}):\n{hit['text']}")



--- SberQuAD ---
Вопрос: От какой фирмы государственного монополиста зависела группа Машина времени до 1991 года?
Эталон: ['Мелодия']
Без контекста: Не знаю.
С семантическим поиском: Фирмы Мелодия.
Исходный документ найден: True

Фрагмент 1 (сходство: 0.872):
на. В этом же году Александр Кутиков становится продюсером группы. Благодаря этому Машина времени больше не зависит от государственного монополиста фирмы Мелодия .

Фрагмент 2 (сходство: 0.856):
К весне 1979 года между двумя участниками группы — А. Макаревичем и С. Кавагоэ, — произошёл конфликт. В книге Всё очень просто сам Макаревич говорит и о творческом кризисе в группе, и о его личных конфликтах с Кавагоэ. Одним из поводов к обиде для Кавагоэ, считавшего себя равноправным основателем группы, стало то что к концу 1970-х Машина времени всё больше воспринималась публикой как группа Андрея Макаревича , а остальные участники, в том числе и он сам, оказались отодвинуты на второй план[3][8

Фрагмент 3 (сходство: 0.849):
В 1982—1984 

В обоих примерах семантический поиск нашел фрагмент с ответом и поставил его на первое место. Для вопроса о группе «Машина времени» модель с контекстом назвала фирму «Мелодия», тогда как без контекста ответила «Не знаю». Для вопроса о трактате «Поэтика» первый фрагмент прямо указывает авторство Аристотеля, но модель дала тот же ответ и без поиска.
Также в обеих выдачах остается много лишних фрагментов. Они относятся к группе «Машина времени» или к теме поэтики, но не содержат сведений, необходимых для ответа. Семантическая близость помогает найти тексты по общей теме, однако не гарантирует их полезность для конкретного вопроса. Чтобы уменьшить количество посторонней информации, далее применим реранкер, который повторно оценит найденные фрагменты.

### Задание 2. Сравнение семантического поиска с BM25

На тех же validation-вопросах из каждого датасета самостоятельно оцените вариант с E5-small и сравните его с результатами BM25 и ответами без контекста.

1. Составьте таблицу с EM, F1 и средним временем для трех вариантов. Для методов поиска дополнительно покажите `hit` и `document_recall`.
2. Разберите несколько примеров, где BM25 и семантический поиск нашли разные фрагменты или дали разные ответы. Есть ли во фрагментах информация, подтверждающая ответ?
3. Сделайте вывод: какой метод показал лучший результат на каждом наборе и как изменилось время ответа? Учитывайте ограничения EM и F1 при сравнении разных форм ответа.


## Реранкер: повторная оценка найденных фрагментов

В примерах семантический поиск поставил нужный фрагмент на первое место, но вместе с ним вернул тексты, которые относятся к теме вопроса и не содержат ответа. Попробуем улучшить порядок выдачи с помощью **реранкера**.

Реранкер не ищет по всей коллекции. Сначала семантический поиск отбирает несколько десятков кандидатов, затем отдельная модель повторно оценивает каждый из них. Получившиеся оценки используются для сортировки, после чего в контекст языковой модели попадают первые `TOP_K` фрагментов.

### Как работает cross-encoder

При семантическом поиске вопрос и фрагмент преобразуются в векторы отдельно. Векторы фрагментов можно вычислить заранее, поэтому поиск по ним удобен для большой коллекции. Реранкер типа **cross-encoder** получает вопрос и текст фрагмента вместе и выдает оценку их соответствия. Совместная обработка позволяет учитывать связь между словами вопроса и фрагмента, но каждую пару нужно обработать отдельно. Поэтому cross-encoder применяют не ко всем документам, а к кандидатам первого этапа.

В этой работе используется модель `BAAI/bge-reranker-v2-m3`. Для вопроса $q$ и фрагмента $d$ обозначим ее оценку как $s(q, d)$. Реранкер вычисляет оценку для каждого кандидата:

$$
s_i = s(q, d_i), \qquad i = 1, \ldots, N,
$$

где $N$ , число найденных кандидатов. Затем фрагменты сортируются по убыванию $s_i$. Большая оценка означает, что модель считает фрагмент более подходящим к вопросу. Эта оценка не обязательно является вероятностью и не гарантирует, что фрагмент действительно содержит ответ.




In [ ]:
class RerankedSearch:
    def __init__(self, base, reranker, candidate_k=CANDIDATE_K, min_score=None):
        self.base, self.reranker = base, reranker
        self.candidate_k, self.min_score = candidate_k, min_score
    def search(self, question, k=TOP_K):
        if not 1 <= k <= self.candidate_k: raise ValueError('k должен быть от 1 до candidate_k')
        candidates = self.base.search(question, self.candidate_k)
        if not candidates: return []
        scores = np.asarray(self.reranker.predict([(question, h['text']) for h in candidates])).reshape(-1)
        indices = np.argsort(-scores, kind='stable')[:k]
        return [{**candidates[i], 'retrieval_score': candidates[i]['score'], 'score': float(scores[i])}
                for i in indices if self.min_score is None or scores[i] >= self.min_score]
reranker = CrossEncoder(RERANKER_MODEL)
reranked_search = RerankedSearch(semantic_search, reranker)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

**Пояснение к результату.**

Здесь создается реранкер. Он получает кандидатов семантического поиска, оценивает каждую пару «вопрос , фрагмент» и меняет порядок результатов. Сам по себе этот вывод подтверждает загрузку весов, но не показывает, улучшилось ли качество: это проверяется в следующем эксперименте.


### Пример ответа с реранкером

Рассмотрим один вопрос из RuBQ, который использовался в примере семантического поиска. Сначала получим кандидатов с помощью E5-small, затем реранкер пересортирует их. Выведем найденные фрагменты, их оценки и ответ модели. Сравните результаты с предыдущей выдачей: поднялись ли фрагменты, содержащие ответ, и остались ли посторонние тексты?


In [40]:
example_rerank = validation_sets['RuBQ'][idx]
question_rerank = example_rerank['question']
hits_rerank = reranked_search.search(question_rerank, k=TOP_K)
answer_rerank = 'Не знаю' if not hits_rerank else ask_llm(prompt_for(question_rerank, hits_rerank))

print('Вопрос:', question_rerank)
print('Эталон:', example_rerank['answers'])
print('Без контекста:', baseline_results['RuBQ'].iloc[idx]['answer'])
print('С реранкером:', answer_rerank)
print('Исходный документ найден:', any(h['document_id'] in example_rerank['document_ids'] for h in hits_rerank))
for i, hit in enumerate(hits_rerank, 1):
    print(f"\nФрагмент {i} (поиск: {hit['retrieval_score']:.3f}; реранкер: {hit['score']:.3f}):\n{hit['text']}")


Вопрос: В каком городе родился Вольфганг Амадей Моцарт?
Эталон: ['Зальцбург']
Без контекста: Аристотель.
С реранкером: Зальцбург
Исходный документ найден: True

Фрагмент 1 (поиск: 0.888; реранкер: 1.000):
Во́льфганг Амадéй Мо́царт (нем. Wolfgang Amadeus Mozart, МФА [ˈvɔlfɡaŋ amaˈdeus ˈmoːtsaʁt]  слушать; полное имя — Иога́нн Хризосто́м Во́льфганг Амадéй Мо́царт; 27 января 1756, Зальцбург — 5 декабря 1791, Вена) — австрийский композитор и музыкант-виртуоз. Один из самых популярных классических композиторов, Моцарт оказал большое влияние на мировую музыкальную культуру. По свидетельству современников, Моцарт обладал феноменальным музыкальным слухом, памятью и способностью к импровизации. Самый моло

Фрагмент 2 (поиск: 0.916; реранкер: 0.999):
Вольфганг Амадей Моцарт родился 27 января 1756 года в Зальцбурге, бывшем тогда столицей Зальцбургского архиепископства, в доме по адресу Гетрайдегассе, 9. Его отец Леопольд Моцарт был скрипачом и композитором в придворной капелле князя-архиепископа 

### Задание 3. Оценка реранкера

На тех же validation-вопросах из каждого датасета самостоятельно оцените семантический поиск с реранкером и сравните его с результатами предыдущих заданий.

1. Составьте таблицу с EM, F1, средним временем, `hit` и `document_recall` для семантического поиска с реранкером и без него.
2. Разберите несколько примеров, где реранкер изменил порядок найденных фрагментов или ответ модели. Какие фрагменты содержат информацию, подтверждающую ответ?
3. Сделайте вывод: улучшилось ли качество поиска и ответа на каждом наборе и как изменилось время? Учитывайте ограничения EM и F1 при сравнении разных форм ответа.
